# 📊 Análise Sociodemográfica e Econômica Nacional via SQL
## Dataset: IBGE Estatísticas Brasil 2023
### Big Data — Análise com Python + SQLite

---

**Objetivo:** Transformar dados brutos do IBGE em inteligência analítica para suporte à tomada de decisões governamentais e macroeconômicas.

**Dataset:** IBGE_Estatisicas_Brasil.csv — Indicadores sociodemográficos por Unidade Federativa (UF).



In [1]:
import pandas as pd
import sqlite3 

In [2]:
df = pd.read_csv('IBGE_Estatisicas_Brasil.csv',sep=';',encoding='latin1',decimal=',')
df.columns = [
    "UF",
    "Codigo",
    "Gentilico",
    "Governador",
    "Capital",
    "Area_km2",
    "Populacao",
    "Densidade_Demografica",
    "Matriculas_Ensino_Fundamental",
    "IDH",
    "Receitas",
    "Despesas",
    "Rendimento_Per_Capita",
    "Frota_Veiculos"
]
regioes = {
    "Acre": "Norte", "Amazonas": "Norte", "Amapá": "Norte",
    "Pará": "Norte", "Rondônia": "Norte", "Roraima": "Norte",
    "Tocantins": "Norte",
    "Alagoas": "Nordeste", "Bahia": "Nordeste", "Ceará": "Nordeste",
    "Maranhão": "Nordeste", "Paraíba": "Nordeste", "Pernambuco": "Nordeste",
    "Piauí": "Nordeste", "Rio Grande do Norte": "Nordeste", "Sergipe": "Nordeste",
    "Distrito Federal": "Centro-Oeste", "Goiás": "Centro-Oeste",
    "Mato Grosso": "Centro-Oeste", "Mato Grosso do Sul": "Centro-Oeste",
    "Espírito Santo": "Sudeste", "Minas Gerais": "Sudeste",
    "Rio de Janeiro": "Sudeste", "São Paulo": "Sudeste",
    "Paraná": "Sul", "Rio Grande do Sul": "Sul", "Santa Catarina": "Sul"
}
df["Regiao"] = df["UF"].map(regioes)
print(df.head())
conexao = sqlite3.connect(':memory:')
df.to_sql('IBGE_Estatisicas_Brasil', conexao, if_exists='replace', index=False)

         UF  Codigo   Gentilico                      Governador     Capital  \
0      Acre      12     acriano          GLADSON DE LIMA CAMELI  Rio Branco   
1   Alagoas      27    alagoano  PAULO SURUAGY DO AMARAL DANTAS      Maceió   
2     Amapá      16   amapaense      CLÉCIO LUÍS VILHENA VIEIRA      Macapá   
3  Amazonas      13  amazonense             WILSON MIRANDA LIMA      Manaus   
4     Bahia      29      baiano        JERÔNIMO RODRIGUES SOUZA    Salvador   

      Area_km2  Populacao  Densidade_Demografica  \
0   164173.429     830018                   5.06   
1    27830.661    3127683                 112.38   
2   142470.762     733759                   5.15   
3  1559255.881    3941613                   2.53   
4   564760.429   14141626                  25.04   

   Matriculas_Ensino_Fundamental    IDH      Receitas      Despesas  \
0                         153015  0.710  6.632883e+06  6.084417e+06   
1                         458782  0.684  1.195044e+07  1.046063e+07   

27

---
## 1. Obtenção, Ingestão e Persistência dos Dados

Nesta etapa realizamos:
1. Leitura do arquivo CSV com tratamento de encoding e separador decimal
2. Renomeação das colunas para nomes simples (sem acentos e caracteres especiais)
3. Adição manual da coluna `Regiao` por mapeamento de dicionário
4. Persistência dos dados em um banco SQLite em memória

## Desafio 1 — Densidade Demográfica por Estado

A **densidade demográfica** indica quantas pessoas vivem, em média, por quilômetro quadrado em cada estado. Ela é calculada dividindo a população total pela área territorial (km²).

Neste desafio, utilizamos SQL para calcular essa métrica diretamente no banco de dados em memória, ordenando os resultados do estado mais denso para o menos denso.

**Conceitos SQL utilizados:**
- `SELECT` para escolher as colunas exibidas
- Expressão aritmética (`/`) para calcular a densidade
- `AS` para renomear a coluna calculada
- `ORDER BY ... DESC` para ordenar do maior para o menor

In [3]:
query1="""
SELECT
    UF,
    Regiao,
    Populacao /  Area_km2 AS Densidade_demografica,
    CASE
        WHEN (Populacao / Area_km2) > 100 THEN 'Alta'
        WHEN (Populacao / Area_km2) >= 20 AND (Populacao / Area_km2) <= 100 THEN 'Média'
        ELSE 'Baixa'
    END AS Classificacao_densidade
FROM IBGE_Estatisicas_Brasil
ORDER BY Densidade_demografica DESC;
"""
resultado1 = pd.read_sql_query(query1,conexao)
display(resultado1)

,UF,Regiao,Densidade_demografica,Classificacao_densidade
0,Distrito Federal,Centro-Oeste,489.062079,Alta
1,Rio de Janeiro,Sudeste,366.971841,Alta
2,São Paulo,Sudeste,178.919225,Alta
3,Alagoas,Nordeste,112.382634,Alta
4,Sergipe,Nordeste,100.737764,Alta
5,Pernambuco,Nordeste,92.374091,Média
6,Espírito Santo,Sudeste,83.206900,Média
7,Santa Catarina,Sul,79.497609,Média
8,Paraíba,Nordeste,70.389253,Média
9,Rio Grande do Norte,Nordeste,62.540316,Média
